# Simple Local DOCX to PDF Converter

This notebook converts all DOCX files to PDF in a local folder, searching recursively through all subfolders.

**Features:**
- Works with local files (no NAS/SMB needed)
- Recursively searches all subfolders for DOCX files
- Converts using LibreOffice, Pandoc, or Python (tries in that order)
- Saves PDFs in the same location as original DOCX files
- Parallel processing for faster conversion
- Skips existing PDFs to avoid re-conversion

**Requirements:**
- For best results, install LibreOffice: `brew install --cask libreoffice`
- Or install Pandoc: `brew install pandoc`
- Python libraries will be installed automatically if needed

## Step 1: Install Required Libraries

In [ ]:
# Install required Python libraries
import subprocess
import sys

def install_package(package):
    try:
        __import__(package.replace('-', '_'))
        print(f"✓ {package} already installed")
    except ImportError:
        print(f"Installing {package}...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", package])
        print(f"✓ {package} installed")

# Required packages
packages = [
    "python-docx", 
    "reportlab"
]

for package in packages:
    install_package(package)

print("\nAll required packages are installed!")

## Step 2: Configuration - Set Your Local Folder Path

In [ ]:
# ========================================
# UPDATE THIS PATH TO YOUR DOCUMENTS FOLDER
# ========================================

import os
from pathlib import Path

# Set your local folder path
# Default is the folder you mentioned
LOCAL_FOLDER_PATH = os.path.expanduser("~/Documents/IRIS DOCS FOR MAVEN")

# Conversion settings
MAX_WORKERS = 3        # Number of parallel conversion workers
SKIP_EXISTING = True   # Skip conversion if PDF already exists

print(f"Local folder: {LOCAL_FOLDER_PATH}")
print(f"Folder exists: {os.path.exists(LOCAL_FOLDER_PATH)}")
print(f"Parallel workers: {MAX_WORKERS}")
print(f"Skip existing PDFs: {SKIP_EXISTING}")

if not os.path.exists(LOCAL_FOLDER_PATH):
    print("\n⚠️  WARNING: The specified folder does not exist!")
    print("Please update LOCAL_FOLDER_PATH with the correct path to your documents folder.")

## Step 3: Import Libraries and Define Functions

In [ ]:
import os
import logging
import subprocess
from pathlib import Path
from typing import List, Tuple
from concurrent.futures import ThreadPoolExecutor, as_completed
from datetime import datetime
import glob

# Configure logging for the notebook
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s [%(levelname)s] %(message)s'
)
logger = logging.getLogger(__name__)

print("Libraries imported successfully!")

In [ ]:
# Conversion method checkers
def check_libreoffice_available() -> bool:
    """Check if LibreOffice is available on the system."""
    try:
        result = subprocess.run(['libreoffice', '--version'], 
                              capture_output=True, text=True, timeout=10)
        return result.returncode == 0
    except (subprocess.TimeoutExpired, FileNotFoundError):
        return False

def check_pandoc_available() -> bool:
    """Check if Pandoc is available on the system."""
    try:
        result = subprocess.run(['pandoc', '--version'], 
                              capture_output=True, text=True, timeout=10)
        return result.returncode == 0
    except (subprocess.TimeoutExpired, FileNotFoundError):
        return False

# Check available methods
libreoffice_available = check_libreoffice_available()
pandoc_available = check_pandoc_available()

print(f"LibreOffice available: {libreoffice_available}")
print(f"Pandoc available: {pandoc_available}")
print("Python conversion (fallback): Always available")

if not libreoffice_available and not pandoc_available:
    print("\n⚠️  For best results, install LibreOffice or Pandoc:")
    print("   brew install --cask libreoffice")
    print("   brew install pandoc")

In [ ]:
# Conversion functions
def convert_docx_to_pdf_libreoffice(docx_path: str, output_dir: str) -> bool:
    """Convert DOCX to PDF using LibreOffice command line."""
    try:
        cmd = [
            'libreoffice',
            '--headless',
            '--convert-to', 'pdf',
            '--outdir', output_dir,
            docx_path
        ]
        
        result = subprocess.run(cmd, capture_output=True, text=True, timeout=60)
        
        if result.returncode == 0:
            logger.info(f"LibreOffice conversion successful: {os.path.basename(docx_path)}")
            return True
        else:
            logger.error(f"LibreOffice conversion failed: {result.stderr}")
            return False
            
    except subprocess.TimeoutExpired:
        logger.error(f"LibreOffice conversion timeout: {os.path.basename(docx_path)}")
        return False
    except Exception as e:
        logger.error(f"LibreOffice conversion error: {e}")
        return False

def convert_docx_to_pdf_pandoc(docx_path: str, pdf_path: str) -> bool:
    """Convert DOCX to PDF using Pandoc."""
    try:
        cmd = [
            'pandoc',
            docx_path,
            '-o', pdf_path
        ]
        
        result = subprocess.run(cmd, capture_output=True, text=True, timeout=60)
        
        if result.returncode == 0:
            logger.info(f"Pandoc conversion successful: {os.path.basename(docx_path)}")
            return True
        else:
            logger.error(f"Pandoc conversion failed: {result.stderr}")
            return False
            
    except subprocess.TimeoutExpired:
        logger.error(f"Pandoc conversion timeout: {os.path.basename(docx_path)}")
        return False
    except Exception as e:
        logger.error(f"Pandoc conversion error: {e}")
        return False

def convert_docx_to_pdf_python(docx_path: str, pdf_path: str) -> bool:
    """Convert DOCX to PDF using pure Python libraries (fallback method)."""
    try:
        from docx import Document
        from reportlab.lib.pagesizes import letter
        from reportlab.platypus import SimpleDocTemplate, Paragraph
        from reportlab.lib.styles import getSampleStyleSheet
        
        # Read DOCX content
        doc = Document(docx_path)
        text_content = []
        
        for paragraph in doc.paragraphs:
            if paragraph.text.strip():
                text_content.append(paragraph.text)
        
        # Create PDF
        pdf_doc = SimpleDocTemplate(pdf_path, pagesize=letter)
        styles = getSampleStyleSheet()
        story = []
        
        for text in text_content:
            # Clean text for reportlab
            clean_text = text.replace('&', '&amp;').replace('<', '&lt;').replace('>', '&gt;')
            para = Paragraph(clean_text, styles['Normal'])
            story.append(para)
        
        pdf_doc.build(story)
        logger.info(f"Python conversion successful: {os.path.basename(docx_path)}")
        return True
        
    except ImportError as e:
        logger.warning(f"Python conversion libraries not available: {e}")
        return False
    except Exception as e:
        logger.error(f"Python conversion error: {e}")
        return False

print("Conversion functions defined!")

In [ ]:
# File operations
def find_docx_files_recursive(base_path: str) -> List[str]:
    """Recursively find all DOCX files in the specified path and all subfolders."""
    docx_files = []
    
    # Use glob to find all .docx files recursively
    pattern = os.path.join(base_path, '**', '*.docx')
    for file_path in glob.glob(pattern, recursive=True):
        # Skip temporary files that start with ~
        if not os.path.basename(file_path).startswith('~'):
            docx_files.append(file_path)
            print(f"Found: {os.path.relpath(file_path, base_path)}")
    
    logger.info(f"Found {len(docx_files)} DOCX files")
    return docx_files

def pdf_exists(pdf_path: str) -> bool:
    """Check if PDF file already exists."""
    return os.path.exists(pdf_path)

print("File operation functions defined!")

In [ ]:
# Main conversion function
def convert_single_docx_file(docx_path: str, conversion_methods: List[str], skip_existing: bool = True) -> Tuple[bool, str]:
    """Convert a single DOCX file to PDF."""
    # Determine PDF path (same directory as DOCX, with .pdf extension)
    pdf_path = os.path.splitext(docx_path)[0] + '.pdf'
    
    # Check if PDF already exists
    if skip_existing and pdf_exists(pdf_path):
        logger.info(f"PDF already exists, skipping: {os.path.basename(pdf_path)}")
        return True, "PDF already exists"
    
    logger.info(f"Converting: {os.path.basename(docx_path)} -> {os.path.basename(pdf_path)}")
    
    # Get the directory for output
    output_dir = os.path.dirname(docx_path)
    
    # Try conversion methods in order
    conversion_successful = False
    last_error = ""
    
    for method in conversion_methods:
        try:
            if method == 'libreoffice':
                if convert_docx_to_pdf_libreoffice(docx_path, output_dir):
                    conversion_successful = True
                    break
            elif method == 'pandoc':
                if convert_docx_to_pdf_pandoc(docx_path, pdf_path):
                    conversion_successful = True
                    break
            elif method == 'python':
                if convert_docx_to_pdf_python(docx_path, pdf_path):
                    conversion_successful = True
                    break
        except Exception as e:
            last_error = f"{method} failed: {e}"
            logger.warning(last_error)
    
    if not conversion_successful:
        return False, f"All conversion methods failed. Last error: {last_error}"
    
    logger.info(f"Successfully converted: {os.path.basename(docx_path)}")
    return True, "Success"

print("Main conversion function defined!")

## Step 4: Find DOCX Files in Your Local Folder

In [ ]:
# Find all DOCX files in the local folder
print(f"Scanning for DOCX files in: {LOCAL_FOLDER_PATH}")

if not os.path.exists(LOCAL_FOLDER_PATH):
    print(f"❌ Folder does not exist: {LOCAL_FOLDER_PATH}")
    print("Please update the LOCAL_FOLDER_PATH in Step 2 with the correct path.")
else:
    print("✓ Folder exists")
    
    # Find DOCX files
    print("\nScanning for DOCX files...")
    docx_files = find_docx_files_recursive(LOCAL_FOLDER_PATH)
    
    print(f"\n📊 Found {len(docx_files)} DOCX files")
    
    if docx_files:
        print("\nFiles found:")
        for i, file_path in enumerate(docx_files[:10]):
            rel_path = os.path.relpath(file_path, LOCAL_FOLDER_PATH)
            print(f"  {i+1:2d}. {rel_path}")
        
        if len(docx_files) > 10:
            print(f"     ... and {len(docx_files) - 10} more files")
    else:
        print("No DOCX files found in the specified folder.")

## Step 5: Run the Conversion

In [ ]:
# Main conversion process
def run_conversion_process():
    print("=" * 60)
    print("Starting DOCX to PDF Conversion")
    print("=" * 60)
    
    start_time = datetime.now()
    
    # Check if folder exists
    if not os.path.exists(LOCAL_FOLDER_PATH):
        error_msg = f"Local folder does not exist: {LOCAL_FOLDER_PATH}"
        print(f"❌ {error_msg}")
        return {"error": error_msg}
    
    # Find all DOCX files
    docx_files = find_docx_files_recursive(LOCAL_FOLDER_PATH)
    
    if not docx_files:
        print("No DOCX files found")
        return {"total_files": 0, "successful": 0, "failed": 0, "skipped": 0}
    
    # Determine available conversion methods
    available_methods = []
    if libreoffice_available:
        available_methods.append('libreoffice')
    if pandoc_available:
        available_methods.append('pandoc')
    available_methods.append('python')  # Always available as fallback
    
    print(f"Using conversion methods: {available_methods}")
    print(f"Starting conversion of {len(docx_files)} files with {MAX_WORKERS} workers")
    
    # Convert files in parallel
    results = {"total_files": len(docx_files), "successful": 0, "failed": 0, "skipped": 0}
    failed_files = []
    
    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
        future_to_file = {
            executor.submit(convert_single_docx_file, docx_file, available_methods, SKIP_EXISTING): docx_file 
            for docx_file in docx_files
        }
        
        for future in as_completed(future_to_file):
            docx_file = future_to_file[future]
            try:
                success, message = future.result()
                file_name = os.path.basename(docx_file)
                if success:
                    if "already exists" in message:
                        results["skipped"] += 1
                        print(f"⏭️  Skipped: {file_name} (PDF already exists)")
                    else:
                        results["successful"] += 1
                        print(f"✅ Converted: {file_name}")
                else:
                    results["failed"] += 1
                    failed_files.append(f"{file_name}: {message}")
                    print(f"❌ Failed: {file_name} - {message}")
            except Exception as e:
                results["failed"] += 1
                file_name = os.path.basename(docx_file)
                failed_files.append(f"{file_name}: {e}")
                print(f"❌ Exception: {file_name} - {e}")
    
    end_time = datetime.now()
    duration = end_time - start_time
    
    # Summary
    print("\n" + "=" * 60)
    print("Conversion Summary")
    print("=" * 60)
    print(f"Duration: {duration}")
    print(f"Total files: {results['total_files']}")
    print(f"✅ Successful: {results['successful']}")
    print(f"⏭️  Skipped (already exist): {results['skipped']}")
    print(f"❌ Failed: {results['failed']}")
    
    if failed_files:
        print("\nFailed files:")
        for failed_file in failed_files[:10]:  # Show first 10 failures
            print(f"  - {failed_file}")
        if len(failed_files) > 10:
            print(f"  ... and {len(failed_files) - 10} more")
    
    results["failed_files"] = failed_files
    results["duration"] = str(duration)
    return results

# Run the conversion
if 'docx_files' in locals() and docx_files:
    conversion_results = run_conversion_process()
    print(f"\nFinal Results: {conversion_results}")
else:
    print("No DOCX files found to convert. Please check your folder path and run the previous cell first.")

## Optional: Convert Specific Files or Change Settings

In [ ]:
# Example: Convert only specific files or change settings

# To convert only files in a specific subfolder:
# LOCAL_FOLDER_PATH = os.path.expanduser("~/Documents/IRIS DOCS FOR MAVEN/specific_subfolder")

# To overwrite existing PDFs:
# SKIP_EXISTING = False

# To use fewer workers (if you have a slower computer):
# MAX_WORKERS = 1

print("Current settings:")
print(f"Folder: {LOCAL_FOLDER_PATH}")
print(f"Skip existing: {SKIP_EXISTING}")
print(f"Workers: {MAX_WORKERS}")

# Uncomment the line below to run conversion with current settings
# conversion_results = run_conversion_process()

## Troubleshooting

If you encounter issues:

1. **Path errors**: Make sure the folder path is correct and exists
2. **Permission errors**: Ensure you have read/write access to the files
3. **Conversion errors**: Install LibreOffice or Pandoc for better conversion quality
4. **File corruption**: Some DOCX files might be corrupted and cannot be converted
5. **Performance**: Reduce MAX_WORKERS if your computer is running slowly

**Installation commands for better conversion:**
```bash
# Install LibreOffice (recommended)
brew install --cask libreoffice

# Install Pandoc (alternative)
brew install pandoc
```

**To change the folder path:**
Update the `LOCAL_FOLDER_PATH` variable in Step 2 and re-run the cells.